# Model Experiments — Ames House Prices

This notebook compares untuned candidate models on one reproducible split. Reusable training and evaluation logic remains in `src/models/`.

## Experiment contract

All candidates train on the same `X_train`/`y_train` partition and are evaluated on the same held-out test set. Their preprocessing is inside each pipeline, so imputers, encoders, and scalers learn only from training rows. No hyperparameters are tuned in this notebook.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path.cwd().resolve()
if not (project_root / 'data').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.data.preprocessing import load_processed_data
from src.models.evaluate import evaluate_models
from src.models.train import create_train_test_split, train_model_comparison_candidates

sns.set_theme(style='whitegrid', palette='deep')

## Train and evaluate candidates

The linear model is a transparent compact benchmark. The tree models receive the broader non-redundant feature set because they can naturally capture non-linear effects and interactions. This is an architectural choice, not hyperparameter tuning.

In [ ]:
data = load_processed_data(project_root / 'data/processed/train_clean.csv')
X_train, X_test, y_train, y_test = create_train_test_split(data)
models = train_model_comparison_candidates(X_train, y_train)
results = evaluate_models(models, X_test, y_test)
results.style.format({'mae': '${:,.0f}', 'rmse': '${:,.0f}', 'r2': '{:.3f}'})

## Error comparison

**What we are analyzing:** typical absolute error and large-error-sensitive RMSE for each candidate.

**Why it matters:** the best R² alone may still have an unacceptably high dollar error or a complexity cost that is not justified for the API.

In [ ]:
plot_data = results.melt(id_vars='Model', value_vars=['mae', 'rmse'], var_name='metric', value_name='dollars')
plt.figure(figsize=(10, 6))
sns.barplot(data=plot_data, x='Model', y='dollars', hue='metric')
plt.title('Prediction error by model (lower is better)')
plt.xlabel('Model')
plt.ylabel('Error in dollars')
plt.xticks(rotation=12)
plt.tight_layout()

## Model-selection discussion

Use the table and chart to assess all of the following before selecting a tuning candidate:

- **Prediction error:** lower MAE and RMSE are preferred.
- **Generalization:** all scores come from identical held-out rows; tuning still must use training-only cross-validation.
- **Complexity:** Linear Regression is the simplest; Random Forest stores many trees; Gradient Boosting trains sequentially.
- **Interpretability:** Linear Regression is most direct; tree models can provide feature importance but are less transparent.

The next phase will tune only the strongest promising tree-based candidate, if its improvement justifies the added complexity.